In [ ]:
#!pip -q install transformers datasets evaluate accelerate torch


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\owzya\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


DeepPavlov/rubert-base-cased

In [ ]:
import pandas as pd

data = pd.read_csv("data.csv", sep="\t")
data["sentiment"] = data["sentiment"].replace({"neautral": "neutral"})
data = data.dropna(subset=["review", "sentiment"]).copy()

print(data.shape)
print(data["sentiment"].value_counts())
data.head()

(90000, 2)
sentiment
negative    30000
neutral     30000
positive    30000
Name: count, dtype: int64


,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


In [ ]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

data["sentiment"] = data["sentiment"].astype(str).str.strip().str.lower()

unknown = set(data["sentiment"].unique()) - set(label2id.keys())
print("Unknown labels:", unknown)

data["label"] = data["sentiment"].map(label2id)
data[["review", "sentiment", "label"]].head()

Unknown labels: set()


,review,sentiment,label
0,качество плохое пошив ужасный (горловина напер...,negative,0
1,"Товар отдали другому человеку, я не получила п...",negative,0
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative,0
3,"товар не пришел, продавец продлил защиту без м...",negative,0
4,"Кофточка голая синтетика, носить не возможно.",negative,0


In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data[["review", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

print(train_df.shape, test_df.shape)

(72000, 2) (18000, 2)


In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_ds, test_ds

(Dataset({
     features: ['review', 'label'],
     num_rows: 72000
 }),
 Dataset({
     features: ['review', 'label'],
     num_rows: 18000
 }))

In [6]:
from transformers import AutoTokenizer

model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_len = 192

def tokenize(batch):
    return tokenizer(
        batch["review"],
        truncation=True,
        padding=False,
        max_length=max_len
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["review"])
test_tok  = test_ds.map(tokenize, batched=True, remove_columns=["review"])

train_tok[0]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

C:\Users\owzya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\owzya\.cache\huggingface\hub\models--DeepPavlov--rubert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

{'label': 2,
 'input_ids': [101, 74216, 24991, 8152, 27418, 1515, 68660, 6636, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
import numpy as np
import evaluate

from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
training_args = TrainingArguments(
    output_dir="rubert_sentiment_runs",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

C:\Users\owzya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\owzya\AppData\Local\Temp\ipykernel_3184\3341249942.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [9]:
trainer.train()

  0%|          | 0/9000 [00:00<?, ?it/s]

{'loss': 1.0268, 'grad_norm': 4.50253963470459, 'learning_rate': 1.988888888888889e-05, 'epoch': 0.01}
{'loss': 0.7268, 'grad_norm': 15.228668212890625, 'learning_rate': 1.977777777777778e-05, 'epoch': 0.02}
{'loss': 0.7247, 'grad_norm': 5.896280288696289, 'learning_rate': 1.9666666666666666e-05, 'epoch': 0.03}
{'loss': 0.6743, 'grad_norm': 8.168959617614746, 'learning_rate': 1.9555555555555557e-05, 'epoch': 0.04}
{'loss': 0.67, 'grad_norm': 7.566402912139893, 'learning_rate': 1.9444444444444445e-05, 'epoch': 0.06}
{'loss': 0.6286, 'grad_norm': 22.67428970336914, 'learning_rate': 1.9333333333333333e-05, 'epoch': 0.07}
{'loss': 0.6223, 'grad_norm': 4.908501148223877, 'learning_rate': 1.9222222222222225e-05, 'epoch': 0.08}
{'loss': 0.6543, 'grad_norm': 9.405413627624512, 'learning_rate': 1.9111111111111113e-05, 'epoch': 0.09}
{'loss': 0.6325, 'grad_norm': 6.912664413452148, 'learning_rate': 1.9e-05, 'epoch': 0.1}
{'loss': 0.6374, 'grad_norm': 10.987982749938965, 'learning_rate': 1.888888

  0%|          | 0/563 [00:00<?, ?it/s]

{'eval_loss': 0.5027967691421509, 'eval_accuracy': 0.7764444444444445, 'eval_f1_macro': 0.7790907489597004, 'eval_runtime': 53.5068, 'eval_samples_per_second': 336.406, 'eval_steps_per_second': 10.522, 'epoch': 1.0}
{'loss': 0.4585, 'grad_norm': 4.1993207931518555, 'learning_rate': 9.88888888888889e-06, 'epoch': 1.01}
{'loss': 0.5067, 'grad_norm': 10.37187385559082, 'learning_rate': 9.777777777777779e-06, 'epoch': 1.02}
{'loss': 0.4658, 'grad_norm': 8.004670143127441, 'learning_rate': 9.666666666666667e-06, 'epoch': 1.03}
{'loss': 0.4363, 'grad_norm': 6.829779624938965, 'learning_rate': 9.555555555555556e-06, 'epoch': 1.04}
{'loss': 0.4716, 'grad_norm': 13.139161109924316, 'learning_rate': 9.444444444444445e-06, 'epoch': 1.06}
{'loss': 0.5095, 'grad_norm': 5.897472858428955, 'learning_rate': 9.333333333333334e-06, 'epoch': 1.07}
{'loss': 0.4617, 'grad_norm': 6.7088470458984375, 'learning_rate': 9.222222222222224e-06, 'epoch': 1.08}
{'loss': 0.4401, 'grad_norm': 9.444256782531738, 'lear

  0%|          | 0/563 [00:00<?, ?it/s]

{'eval_loss': 0.504026472568512, 'eval_accuracy': 0.783, 'eval_f1_macro': 0.7856517694675414, 'eval_runtime': 61.5748, 'eval_samples_per_second': 292.327, 'eval_steps_per_second': 9.143, 'epoch': 2.0}
{'train_runtime': 1733.205, 'train_samples_per_second': 83.083, 'train_steps_per_second': 5.193, 'train_loss': 0.5098138565487332, 'epoch': 2.0}


TrainOutput(global_step=9000, training_loss=0.5098138565487332, metrics={'train_runtime': 1733.205, 'train_samples_per_second': 83.083, 'train_steps_per_second': 5.193, 'total_flos': 7899832365985728.0, 'train_loss': 0.5098138565487332, 'epoch': 2.0})

In [10]:
metrics = trainer.evaluate()
metrics

  0%|          | 0/563 [00:00<?, ?it/s]

{'eval_loss': 0.504026472568512,
 'eval_accuracy': 0.783,
 'eval_f1_macro': 0.7856517694675414,
 'eval_runtime': 55.7022,
 'eval_samples_per_second': 323.147,
 'eval_steps_per_second': 10.107,
 'epoch': 2.0}

In [11]:
from sklearn.metrics import classification_report

pred = trainer.predict(test_tok)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)

print(classification_report(y_true, y_pred, target_names=[id2label[0], id2label[1], id2label[2]]))

  0%|          | 0/563 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       0.78      0.72      0.75      6000
     neutral       0.66      0.75      0.70      6000
    positive       0.93      0.87      0.90      6000

    accuracy                           0.78     18000
   macro avg       0.79      0.78      0.79     18000
weighted avg       0.79      0.78      0.79     18000



In [12]:
save_dir = "rubert_sentiment_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)

Saved to: rubert_sentiment_model


In [14]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

def predict_rubert(text):

    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    cls = int(torch.argmax(out.logits, dim=-1).item())

    return id2label[cls]

In [16]:
predict_rubert("Хорошее качество, рекомендую!")

'positive'

In [25]:
predict_rubert("Качество ужасное, одежда плохая и рвется")

'negative'

In [30]:
predict_rubert("Качество так себе")

'neutral'

In [31]:
import random

for i in range(10):
    
    text = random.choice(test_df["review"].tolist())
    
    pred = predict_rubert(text)
    
    print("Текст:", text[:120])
    print("Предсказание:", pred)
    print()

Текст: На 42 S подошёл идеально. Цена соответствует качеству! 
Предсказание: positive

Текст: товар после таможни не отслеживался на почту так и не пришол.После срока защиты покупателя оплату провели спор пропал. в
Предсказание: negative

Текст: максимальный размер:44! ткань в рубчик,смотрится не так как на фото... расстроена,очень на фото оно мне понравилось....(
Предсказание: neutral

Текст: очень долгая доставка  и не отслеживалась
Предсказание: neutral

Текст: Товар не советую, чулки маленькие, резинок нет, а значит их носить нельзя, очень тонкие, порвутся как только будите одев
Предсказание: negative

Текст: отличная кофточка,лёгкая,тонкая!точно как на картинке!размерная сетка соответствует.
Предсказание: positive

Текст: Заказ не пришёл, продавец неохотно вернул деньги
Предсказание: negative

Текст: Имеются дырки на швах, и не маленькие 
Предсказание: neutral

Текст: Товар пришёл не тот который заказывала, заказала чёрный с, пришёл красный  другого размера, да ещё и дырявый
Предс